In [26]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

from xgboost import XGBClassifier

import optuna

In [27]:
df = pd.read_csv("datasets/kingametric_credit_risk.csv")

In [28]:
df.shape

(8744, 45)

In [31]:
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.fillna(df.median(numeric_only=True), inplace=True)

In [32]:
cat_cols = ["Payment_of_Min_Amount", "Credit_Mix", "Payment_Behaviour", "Borrower_Tier"]

df[cat_cols] = df[cat_cols].astype("category")

In [33]:
X = df.drop(columns=["Default_Flag"], axis=1)
y = df["Default_Flag"]

In [34]:
X.shape

(8744, 44)

In [36]:
corr = X.corr(numeric_only=True).abs()

upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

to_drop = [col for col in upper.columns if any(upper[col] > 0.85)]

X = X.drop(columns=to_drop)

In [37]:
X.shape

(8744, 35)

In [38]:
folds = StratifiedKFold(n_splits=5, shuffle=True,random_state=42)

In [39]:
base_model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    enable_categorical=True, 
    tree_method="hist",
    learning_rate=0.05,
    random_state=42
)
base_model.fit(X, y)

importances = base_model.feature_importances_

feature_importance = (
    pd.Series(importances, index=X.columns).sort_values(ascending=False)
)

In [40]:
top_features = feature_importance.head(20).index

X_select = X[top_features]

In [ ]:
def objective(trial):
    
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 300, 800),
        "max_depth": trial.suggest_int("max_depth", 3, 6),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1),
        "subsample": trial.suggest_float("subsample", 0.7, 0.95),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 0.95),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0, 0.5),
        "reg_alpha": trial.suggest_float("reg_alpha", 0, 1),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.5, 3),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1, 5),
        "eval_metric": "auc",
        "tree_method": "hist",
        "random_state": 42
    }